## Notebook10

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
us_pop = (
    pl.read_csv(ub + "data/us_city_population.csv")
    .select(c.city, c.state, c.year, c.population)
    .filter(c.population.null_count().over(c.city) <= 1)
)

### Questions

This week's data is the population of 289 American cities at every census from
1790 to 2010, which makes 6,647 rows. Each row is one city in one census year.
The `population` column is in thousands, so New York City in 2010 is 8175.133 and
not 8,175,133. A city that did not exist yet has a population of 0 rather than a
missing value, which matters more than it sounds like it should.

The cities are roughly the largest ones in the country today, traced backward
through time. That means the table is not a sample of American cities and it is
certainly not all of them: 46 states appear, some of them with a single city.
Keep that in mind whenever you compute a total, because the total is over the
cities that happen to be here.

Unless a question says otherwise, just print the result of each block; do not save
it to a variable.

1. Start with the tools you already have. For the 2010 census only, compute each
state's total city population and the number of cities it contributes, and sort
the states from largest to smallest.

In [ ]:
(
    us_pop
    .filter(c.year == 2010)
    .group_by(c.state)
    .agg(
        pop_total = c.population.sum(),
        n = pl.len()
    )
    .sort(c.pop_total, descending=True)
)

California leads with 71 cities and 18.2 million people in them. Now suppose you
want each city's share of its own state's total. The share needs two numbers on
the same row: the city's population, which you have, and the state total, which is
sitting in a different table with one row per state. Aggregation put it there and
aggregation cannot get it back.

2. `.over()` can. Take the 2010 census, write each state's total onto every city in
that state, and compute the share. Sort by share, largest first.

In [ ]:
(
    us_pop
    .filter(c.year == 2010)
    .with_columns(
        state_total = c.population.sum().over(c.state),
        share = c.population / c.population.sum().over(c.state) * 100
    )
    .sort(c.share, descending=True)
)

The top of the result is a column of 100.0. Anchorage is 100 percent of Alaska,
Little Rock is 100 percent of Arkansas, and so on down a list of states that
contribute exactly one city to the table. This is not a finding about Alaska. It
is question 1's `n` column coming back to haunt you: the share is a share of the
cities in *this table*, and for those states the table holds one city.

3. So look at the states that have enough cities for the share to mean anything.
Keep only the states with at least five cities in the table, then compute the same
share. The count is itself a window: `pl.len().over(c.state)` gives every row the
number of rows in its state, and a window expression can go straight inside a
`.filter()`.

In [ ]:
(
    us_pop
    .filter(c.year == 2010)
    .filter(pl.len().over(c.state) >= 5)
    .with_columns(
        share = c.population / c.population.sum().over(c.state) * 100
    )
    .sort(c.share, descending=True)
)

New York City is 90 percent of New York's urban population and Chicago is 72
percent of Illinois's. Nothing is added to the table in the filter and nothing is
removed from it in the `with_columns`. The same expression does different work
depending on which context you drop it into.

4. To see the difference between the two contexts, compute one number both ways.
Take the mean city population in 2010 within each state, first with `.agg()` and
then with `.over()`.

In [ ]:
(
    us_pop
    .filter(c.year == 2010)
    .group_by(c.state, maintain_order=True)
    .agg(pop_avg = c.population.mean())
)

In [ ]:
(
    us_pop
    .filter(c.year == 2010)
    .with_columns(pop_avg = c.population.mean().over(c.state))
    .select(c.city, c.state, c.population, c.pop_avg)
)

**Answer**:
rows, one per state, and the cities are gone. `.over()` returns all 289 cities with
the state average copied onto each one. Only the second result can answer a question
that compares a city to its state, because only the second result still has the
cities in it. Aggregation reduces rows and window functions preserve them.

5. Here is the comparison that needs them preserved. Keep the 2010 cities whose
population is above the mean city population of their own state.

In [ ]:
(
    us_pop
    .filter(c.year == 2010)
    .filter(c.population > c.population.mean().over(c.state))
    .sort(c.state)
)

Seventy-six cities of 289, which is barely a quarter of them. That is what a mean
does when one city in the state is enormous: Los Angeles drags California's average
up past most of the state, and the average stops describing a typical city. Swap
`.mean()` for `.median()` and 132 cities come back instead, which is much nearer to
half, because that is what a median is for.

6. Every window so far has grouped by state and stayed inside one year. Turn it
around: group by *year* and look across the states. Give each city its share of all
289 cities' population in its census year, then follow New York City down the
column.

In [ ]:
(
    us_pop
    .with_columns(share = c.population / c.population.sum().over(c.year) * 100)
    .filter(c.city == "New York City, NY")
    .sort(c.year)
    .select(c.year, c.population, c.share)
)

New York has never stopped growing and its share has been falling since 1920, from
17.7 percent of these cities to 9.6 percent in 2010. Both statements come from the
same two columns. The city got bigger; the country's other cities got bigger faster.

7. Now for the ordered windows. Sort the table by city and year, and compute how
much each city gained since the previous census by subtracting the shifted column.
Do it without an `.over()` first, then print the 1790 rows.

In [ ]:
(
    us_pop
    .sort(c.city, c.year)
    .with_columns(change = c.population - c.population.shift(1))
    .filter(c.year == 1790)
    .sort(c.change)
)

**Answer**:
remarkable thing for a town of zero people to do. The row above Newark's 1790 row is
not Newark. It is the 2010 row of the city that sorts just before it, so the shift
walked straight across the boundary and subtracted New York City's modern population
from a settlement that did not exist yet. Every city's first row is broken this way,
288 of them, and the only clean row in the result is the null at the top: Abilene
sorts first in the table, so it really does have nothing above it.

8. Fix it with `.over(c.city)`, which tells the shift to restart at each city. Then
drop the nulls, sort by the change, and see which decade of which city added the
most people.

In [ ]:
(
    us_pop
    .sort(c.city, c.year)
    .with_columns(change = c.population - c.population.shift(1).over(c.city))
    .drop_nulls(c.change)
    .sort(c.change, descending=True)
)

The first census of every city now has a `null` change instead of a nonsense one,
because there is no row above it to subtract, and that is why the `.drop_nulls()` is
there: without it Polars sorts those nulls to the top of a descending sort and you
read a screen of blanks. What is left is New York City adding 1.9 million people
between 1890 and 1900, then 1.3 million more in the next ten years, with Chicago,
Los Angeles, and Detroit arriving in the 1920s. Nine of the ten largest gains happen
between 1890 and 1930. The tenth is New York again, in the 1990s, which is the only
row in the top of this list that is not the story of the industrial city.

9. Sort the other way and you get the second half of that story. Keep only the 2010
rows with a negative change, so the question becomes: which cities lost people
between 2000 and 2010?

In [ ]:
(
    us_pop
    .sort(c.city, c.year)
    .with_columns(change = c.population - c.population.shift(1).over(c.city))
    .filter(c.year == 2010, c.change < 0)
    .sort(c.change)
)

Sixty-one cities shrank in that decade. Detroit lost 237,000 people and Chicago
200,000, which is the long story. New Orleans lost 141,000, which is not: that is
one hurricane, in August 2005, landing in a column of census counts taken five years
on either side of it. A decade is a coarse instrument and it does not tell you which
of these two kinds of loss you are looking at.

10. Absolute change favors big cities, so try growth as a percentage instead. Use
the same shifted column as a denominator, sort with the fastest growth first, and
look at what comes back.

In [ ]:
(
    us_pop
    .sort(c.city, c.year)
    .with_columns(
        growth = (c.population / c.population.shift(1).over(c.city) - 1) * 100
    )
    .drop_nulls(c.growth)
    .sort(c.growth, descending=True)
)

**Answer**:
yet has a population of 0, and dividing by that 0 is the problem: 0 divided by 0 is
`NaN`, and any real population divided by 0 is `inf`. Both are floats, so neither is
a `null`, so `.drop_nulls()` did not touch them, and Polars sorts `NaN` above every
real number in a descending sort. There are 2,296 `NaN` values, one for every pair of
consecutive censuses that both recorded a zero, and 316 `inf` values, one for every
time a city goes from a zero count to a real one. This is the distinction from Chapter 3
arriving with a bill: missing and not-a-number are different things, and only one of
them is a `null`.

11. Percent growth is only defined once a city has people in it, so the obvious
repair is to throw the zeros out. Filter to the rows with a population above zero,
then compute the shift on what remains.

In [ ]:
(
    us_pop
    .sort(c.city, c.year)
    .filter(c.population > 0)
    .with_columns(
        growth = (c.population / c.population.shift(1).over(c.city) - 1) * 100
    )
    .drop_nulls(c.growth)
    .sort(c.growth, descending=True)
)

No `NaN`, no `inf`, and a clean answer at the top: Mesquite, Texas grew 27,426
percent, and the row says 1960. Before you believe it, print Mesquite's history from
1900 on and work out which two censuses that number is actually comparing.

In [ ]:
(
    us_pop
    .filter(c.city == "Mesquite, TX", c.year >= 1900)
    .sort(c.year)
)

**Answer**:
censuses of 0.0, then 27,526 people in 1960. The filter deleted those five rows, and
deleting a row changes what `.shift(1)` means, because "the row above" is a fact
about the table you are holding and not about the calendar. So the growth of the
1950s was computed across sixty years. The zeros are also less innocent than they
looked: population is recorded in thousands to one decimal, so a real town of forty
people is stored as 0.0, and a `0.0` in this column means either "no city here" or
"a village too small to round up," with no way to tell which. Forty-four cities in
the table go from a positive count back to 0.0 at some later census, which no city
has ever actually done.

The repair is to compute the window on the full table and filter afterward, so that
every row still compares itself to the census immediately before it and only the rows
with a zero denominator are dropped.

In [ ]:
(
    us_pop
    .sort(c.city, c.year)
    .with_columns(prev = c.population.shift(1).over(c.city))
    .filter(c.prev > 0)
    .with_columns(growth = (c.population / c.prev - 1) * 100)
    .sort(c.growth, descending=True)
)

Warren, Michigan: 700 people in 1950, 89,246 in 1960, a 12,649 percent decade. That
one is real, and it is what a Detroit suburb looked like when Detroit was still
growing. Percent growth rewards a small denominator, so it hands you villages that
became suburbs and it will never once mention New York, while question 8 hands you
nothing but New York. Both are correct. Choose the one that matches the question you
were asked, and notice that the filter had to come after the window and not before.

12. One city in this table has an actual hole in it. Portland, Oregon has no
population recorded for 1850. Sort by city and year, then use `.forward_fill()` and
`.backward_fill()`, both scoped with `.over(c.city)`, to fill the gap from above and
from below, and look at Portland from 1830 to 1870.

In [ ]:
(
    us_pop
    .sort(c.city, c.year)
    .with_columns(
        fill_fwd = c.population.forward_fill().over(c.city),
        fill_bwd = c.population.backward_fill().over(c.city)
    )
    .filter(c.city == "Portland, OR", c.year.is_between(1830, 1870))
)

Forward fill copies 1840 down and says Portland had nobody in it. Backward fill
copies 1860 up and says it had 2,874 people. The census counted 821. Both fills are
guesses, they are wrong in opposite directions, and the true value sits between them,
which is the most you can generally expect from a fill. Note also that the `.over()`
changed nothing here, since the row above Portland's gap happens to be Portland. It
would have changed everything if the gap had landed on a city's first row, and you do
not get to know that in advance.